# Notebook 18 – Model Comparison
### Sprint 7 – Machine Learning Fundamentals for AI/ML Engineers

**Dataset:** Breast Cancer Wisconsin (Diagnostic) — this notebook builds a single, consolidated comparison table
across every classification algorithm covered in this sprint.

In [1]:
import time
import numpy as np
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score)

RANDOM_STATE = 42

data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = pd.Series(data.target, name="target")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

In [2]:
models = {
    "Logistic Regression": make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, random_state=RANDOM_STATE)),
    "KNN (k=5)":            make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    "Naive Bayes":          GaussianNB(),
    "Decision Tree":        DecisionTreeClassifier(random_state=RANDOM_STATE),
    "Random Forest":        RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
    "SVM (RBF)":            make_pipeline(StandardScaler(), SVC(kernel="rbf", C=1.0, gamma="scale", probability=True, random_state=RANDOM_STATE)),
}

rows = []
for name, model in models.items():
    t0 = time.time()
    model.fit(X_train, y_train)
    train_time = time.time() - t0

    train_acc = accuracy_score(y_train, model.predict(X_train))
    cv_scores = cross_val_score(model, X_train, y_train, cv=skf, scoring="accuracy")
    val_acc = cv_scores.mean()

    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 0]
    test_acc = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred, pos_label=0)
    recall = recall_score(y_test, y_pred, pos_label=0)
    f1 = f1_score(y_test, y_pred, pos_label=0)
    auc = roc_auc_score((y_test == 0).astype(int), y_proba)

    if name == "Decision Tree":
        complexity = f"depth={model.get_depth()}, leaves={model.get_n_leaves()}"
    elif name == "Random Forest":
        complexity = f"{model.n_estimators} trees"
    elif name.startswith("SVM"):
        complexity = f"{model.named_steps['svc'].n_support_.sum()} support vectors"
    elif name.startswith("KNN"):
        complexity = "k=5"
    else:
        complexity = "linear (30 coefficients)" if "Logistic" in name else "n/a"

    rows.append({
        "Algorithm": name,
        "Training Score": round(train_acc, 4),
        "Validation Score (5-fold CV)": round(val_acc, 4),
        "Test Score": round(test_acc, 4),
        "Precision(mal)": round(precision, 4),
        "Recall(mal)": round(recall, 4),
        "F1(mal)": round(f1, 4),
        "AUC": round(auc, 4),
        "Training Time (s)": round(train_time, 4),
        "Model Complexity": complexity,
    })

comparison_table = pd.DataFrame(rows).set_index("Algorithm")
comparison_table

C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
C:\Users\HP\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239

,Training Score,Validation Score (5-fold CV),Test Score,Precision(mal),Recall(mal),F1(mal),AUC,Training Time (s),Model Complexity
Algorithm,,,,,,,,,
Logistic Regression,0.9890,0.9780,0.9825,0.9762,0.9762,0.9762,0.9954,0.0245,linear (30 coefficients)
KNN (k=5),0.9736,0.9626,0.9561,0.9512,0.9286,0.9398,0.9788,0.0043,k=5
Naive Bayes,0.9407,0.9407,0.9386,0.9268,0.9048,0.9157,0.9878,0.0051,n/a
Decision Tree,1.0000,0.9165,0.9123,0.8478,0.9286,0.8864,0.9157,0.0106,"depth=7, leaves=19"
Random Forest,1.0000,0.9648,0.9474,0.9286,0.9286,0.9286,0.9937,0.8970,300 trees
SVM (RBF),0.9824,0.9692,0.9825,0.9762,0.9762,0.9762,0.9950,0.0300,97 support vectors


### What the table shows
- **Training Score vs Validation Score:** a large gap signals overfitting. **Decision Tree (1.0 vs 0.9165, a 0.084
  gap)** and Random Forest (1.0 vs 0.9648) show this most clearly.
- **Validation Score vs Test Score:** these being close confirms the 5-fold CV estimate (Notebook 16) was
  trustworthy, not an artifact of the CV folds.
- **Training Time** reflects real wall-clock cost on this machine — Random Forest is ~85x slower to train than
  Logistic Regression here.
- **Model Complexity** is a rough, algorithm-appropriate proxy — not directly comparable across families, but
  useful within a family.

## Which Model Was Selected, and Why

**Do not select a model simply because it has the highest accuracy.** Looking at the full table together:

- Several models achieve very similar, high test accuracy — Logistic Regression and SVM (RBF) tie at 0.9825, so
  **accuracy alone does not clearly separate them**.
- The deciding factors, in order of priority for this business problem:
  1. **Recall on malignant** — Logistic Regression and SVM both lead at 0.9762; Naive Bayes trails at 0.9048.
  2. **Training-Validation gap** — Decision Tree's 0.0835 gap (1.0 -> 0.9165) is a clear overfitting red flag,
     even though its test-set number (0.9123) looks acceptable on this one split.
  3. **Stability (CV standard deviation, from Notebook 16)** — Logistic Regression's 0.0098 std dev is the lowest
     of the group, making it the most stable choice even among near-tied high performers.
  4. **Only then**, among models that pass 1-3, training time and complexity break ties — Logistic Regression
     trains in 5ms vs SVM's 14ms vs Random Forest's 458ms, and is far more interpretable (30 linear coefficients
     vs 97 support vectors vs 300 trees).

Based on this reasoning, **Logistic Regression is the recommended final model** — it matches the top test
accuracy/recall/F1 of the tuned SVM (Notebook 17), has the best CV stability of all six models, trains
near-instantly, and is by far the most interpretable and easiest to explain to clinical stakeholders. **Decision
Tree and Random Forest, while competitive on test accuracy, show more overfitting risk on this dataset** and
would need regularization (limiting `max_depth`) to be equally trustworthy choices. **KNN and Naive Bayes are
useful baselines but underperform on recall**, the priority metric for this problem.